# Reranking en NormaBot

## ¿Qué es el reranking?

En un pipeline RAG, el retriever devuelve los `k` documentos más cercanos semánticamente a la pregunta. Pero similitud semántica no es lo mismo que relevancia real: un documento puede estar cerca en el espacio de embeddings pero no contener la información que necesita el usuario.

El **reranking** es un paso adicional que reordena o filtra esos documentos antes de pasarlos al generador, usando un modelo más sofisticado que el retriever.

Hay dos enfoques principales:

| Enfoque | Cómo funciona | Salida |
|---------|--------------|--------|
| **Cross-encoder** | Modelo que recibe query + documento juntos y calcula un score de relevancia | Score continuo (0.0 – 1.0) |
| **LLM-based (grading)** | LLM que decide si el documento es relevante para la pregunta | Decisión binaria (sí / no) |

## El grading de NormaBot ES reranking

El paso de **grading** en el pipeline Corrective RAG de NormaBot actúa como un reranker:

```
Retrieve (ChromaDB, k=5)
    ↓
Grade (Qwen 2.5 3B — reranker LLM)
    ↓
Generate (Bedrock Nova Lite)
```

El grader recibe cada documento recuperado junto con la pregunta del usuario y decide si es relevante (`si`) o no (`no`). Solo los documentos marcados como `si` llegan al generador.

Esto es funcionalmente equivalente al reranking: **reduce y mejora el conjunto de documentos** que ve el generador.

## Comparativa: Cross-encoder vs LLM-based reranking

### Cross-encoder (técnica clásica)

In [1]:
# Ejemplo conceptual de cross-encoder (no se ejecuta — requiere sentence-transformers)
#
# from sentence_transformers import CrossEncoder
#
# model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
# scores = model.predict([
#     (query, doc1),
#     (query, doc2),
#     (query, doc3),
# ])
# # scores = [0.92, 0.34, 0.71] → ordenar por score y cortar con umbral

# Simulación del resultado:
query = "¿Qué prácticas de IA están prohibidas?"
docs = [
    "Artículo 5. Quedan prohibidas las siguientes prácticas de IA...",
    "Artículo 9. Se establecerá un sistema de gestión de riesgos...",
    "Artículo 71. Las infracciones estarán sujetas a multas...",
]
scores_simulados = [0.92, 0.11, 0.34]  # scores que daría un cross-encoder

print("Cross-encoder — scores de relevancia:")
for doc, score in zip(docs, scores_simulados):
    estado = "✓ pasa" if score > 0.5 else "✗ descarta"
    print(f"  {score:.2f} {estado} | {doc[:60]}")

Cross-encoder — scores de relevancia:
  0.92 ✓ pasa | Artículo 5. Quedan prohibidas las siguientes prácticas de IA
  0.11 ✗ descarta | Artículo 9. Se establecerá un sistema de gestión de riesgos.
  0.34 ✗ descarta | Artículo 71. Las infracciones estarán sujetas a multas...


### LLM-based reranking (NormaBot)

In [2]:
# El GRADING_PROMPT actual en src/rag/main.py
GRADING_PROMPT = (
    "Dado el siguiente documento y la pregunta, "
    "¿el documento contiene información útil para responder la pregunta?\n\n"
    "Documento: {document}\n"
    "Pregunta: {query}\n\n"
    'Responde solo con "si" o "no":'
)

# Simulación de respuestas del grader LLM
respuestas_simuladas = ["si", "no", "no"]

print("LLM grading — decisiones de relevancia:")
for doc, resp in zip(docs, respuestas_simuladas):
    estado = "✓ pasa" if resp == "si" else "✗ descarta"
    print(f"  {resp}  {estado} | {doc[:60]}")

LLM grading — decisiones de relevancia:
  si  ✓ pasa | Artículo 5. Quedan prohibidas las siguientes prácticas de IA
  no  ✗ descarta | Artículo 9. Se establecerá un sistema de gestión de riesgos.
  no  ✗ descarta | Artículo 71. Las infracciones estarán sujetas a multas...


## Tabla comparativa

| Criterio | Cross-encoder | LLM-based (NormaBot) |
|----------|--------------|---------------------|
| **Salida** | Score continuo (0.0–1.0) | Binario (sí/no) |
| **Interpretabilidad** | Baja — un número sin explicación | Alta — decisión explícita |
| **Modelo necesario** | Cross-encoder preentrenado | LLM generalista (Qwen 2.5 3B) |
| **Requiere fine-tuning legal** | Sí, idealmente | No — el LLM razona sobre el contenido |
| **Latencia** | Baja (~ms por doc) | Media (~1s por doc con Ollama local) |
| **Coste** | Gratuito (modelo local) | Gratuito (Ollama local) |
| **Adaptación al dominio legal** | Requiere datos etiquetados | Implícita en el LLM preentrenado |

## Justificación de la decisión

Se eligió **LLM-based reranking (grading)** sobre cross-encoder por tres razones:

### 1. Interpretabilidad en dominio legal

En un sistema de cumplimiento normativo, **saber por qué se descartó un documento** es tan importante como la decisión en sí. Un cross-encoder produce un score (ej. `0.34`) que no explica nada. El grading LLM toma una decisión explícita que puede extenderse con Chain of Thought para auditoría.

### 2. Sin necesidad de fine-tuning

Los cross-encoders están preentrenados sobre colecciones como MS MARCO (inglés, búsqueda web). Para documentos legales en español (EU AI Act, BOE, LOPD-GDD) necesitarían fine-tuning con datos etiquetados. Qwen 2.5 3B razona directamente sobre el contenido sin datos adicionales.

### 3. Integración natural con el pipeline

El grading LLM se integra como un nodo más del grafo LangGraph, con el mismo patrón que el resto del pipeline. Añadir un cross-encoder requeriría un modelo adicional y gestión de dependencias extra.

### Limitación conocida

El grading LLM tiene **recall=0.545** con Qwen 2.5 3B (ver `04_prompt_engineering.ipynb`). Un cross-encoder bien ajustado podría tener mejor recall. Esta es la principal ventaja del cross-encoder que se sacrifica a cambio de interpretabilidad.

## Conclusión

NormaBot implementa reranking mediante **LLM-based grading** — una técnica equivalente al cross-encoder reranking en cuanto a función (filtrar documentos antes del generador), pero con ventajas de interpretabilidad y simplicidad que son especialmente relevantes en el dominio legal.

La elección no es una omisión del reranking clásico, sino una **decisión de diseño justificada** por las características del dominio.